# 23N Field Tests #02 — Small Model, Big Factory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft02-small-model-defects/ft02_small_model_defects.ipynb)

**The question.** Can a *small* model running on a cheap CPU box catch manufacturing defects as well as a large one, at a
fraction of the cost? We train a PatchCore-style anomaly detector on photos of **good parts only** (metal nuts, screws,
glass bottles, power cables), measure how well it separates defective from good parts (image-level AUROC), time it on a
plain CPU (our proxy for an edge box on the shop floor), and turn that latency into **₹ per 1,000 inspections**. We
compare a small backbone (ResNet-18, ~11M params) against a large one (WideResNet-50, ~69M params) and, optionally, a
frontier vision-language model called zero-shot through an API.

**Dataset.** [MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad) (Bergmann et al., CVPR 2019 / IJCV 2021),
downloaded per-category from MVTec's official links. **Licence: CC BY-NC-SA 4.0 — non-commercial use only.** This notebook
is an independent research study on public data; it is not affiliated with MVTec or any manufacturer.

**Runtime.** Free Colab T4 GPU: ~12–18 min for "Run all" (about a third of that is downloading ~1 GB). CPU-only runtime:
~20–25 min (smaller memory bank; see `CORESET_RATIO`). No API key needed; the frontier-model comparison is an optional
cell that skips itself when `ANTHROPIC_API_KEY` is not set.

*23N AI Labs · Gandhinagar, India · 23nlabs.com*

## 1 · Setup

In [ ]:
# torch, torchvision, scikit-learn, pandas, matplotlib are preinstalled on Colab and Kaggle.
%pip install -q gradio anthropic

In [ ]:
import os, sys, io, re, time, math, json, glob, base64, random, shutil, tarfile, subprocess, urllib.request
import numpy as np
import pandas as pd
import torch, torchvision
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from sklearn.metrics import roc_auc_score

SEED = 23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- data ---------------------------------------------------------------------------------
# Four MVTec AD categories chosen for Indian manufacturing relevance:
# fasteners (nuts, screws), glass packaging, and the wire & cable industry.
CATEGORIES = ["metal_nut", "screw", "bottle", "cable"]
CATEGORY_LABELS = {"metal_nut": "Metal nuts", "screw": "Screws", "bottle": "Glass bottles", "cable": "Power cables"}
DATA_ROOT = os.environ.get("MVTEC_ROOT", "data/mvtec_ad")
OUT_DIR = "ft02_outputs"; os.makedirs(OUT_DIR, exist_ok=True)

# ---- method -------------------------------------------------------------------------------
BACKBONES = {"small": "resnet18", "large": "wide_resnet50_2"}
BACKBONE_NAMES = {"resnet18": "ResNet-18", "wide_resnet50_2": "WideResNet-50"}
BACKBONE_WEIGHTS = "DEFAULT"          # ImageNet-pretrained torchvision weights
RESIZE, IMG_SIZE = 256, 224           # resize then centre-crop (standard PatchCore preprocessing)
CORESET_RATIO = 0.10 if DEVICE == "cuda" else 0.01   # fraction of training patches kept in the memory bank
CALIB_FRACTION = 0.15                 # share of GOOD training images held out to set the alarm threshold
THRESH_QUANTILE = 0.99                # threshold = this quantile of held-out good-part scores
BATCH_SIZE = 16
LATENCY_N = 20                        # images timed per category for latency
CPU_THREADS = os.cpu_count() or 2

# ---- cost assumptions (ALL ESTIMATES, edit to your situation) ---------------------------
CPU_BOX_COST_INR = 45_000             # fanless mini-PC, 4-core x86, 16 GB RAM (estimate, Nov 2026)
CPU_BOX_LIFE_YEARS = 3                # straight-line amortisation
CPU_BOX_HOURS_PER_DAY = 16            # two shifts
CPU_BOX_POWER_W = 25                  # average draw under load (estimate)
TARIFF_INR_PER_KWH = 8.0              # industrial electricity tariff (estimate)
CLOUD_GPU_INR_PER_HOUR = 50.0         # T4-class cloud GPU instance (estimate, ~US$0.55/h)
USD_TO_INR = 88.0                     # exchange rate (estimate)

# ---- OPTIONAL frontier-model comparison (skipped without ANTHROPIC_API_KEY) -------------
CLAUDE_MODEL = "claude-sonnet-5-5"
CLAUDE_PRICE_IN_USD_PER_MTOK = 2.0    # list price at time of writing; check anthropic.com/pricing
CLAUDE_PRICE_OUT_USD_PER_MTOK = 10.0
CLAUDE_SAMPLE_N = 40                  # balanced good/defective sample across categories
CLAUDE_MAX_SIDE = 768                 # images are downscaled before upload

# ---- brand ----------------------------------------------------------------------------------
BG, FG, ACCENT, TEAL, MUTED = "#0B0F14", "#F5F7FA", "#FF7A1A", "#2EC4B6", "#8A94A6"
FONT = "DejaVu Sans"
matplotlib.rcParams["font.family"] = FONT

print(f"torch {torch.__version__} | device: {DEVICE} | CPU threads: {CPU_THREADS}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("ffmpeg:", shutil.which("ffmpeg") or "NOT FOUND (video cell will fail)")

## 2 · Data — MVTec AD (4 categories, ~970 MB)

Per-category archives from MVTec's official download page. No login or form is needed for these links. If you already
have MVTec AD (for example as a Kaggle dataset input), set `MVTEC_ROOT` or attach it: the cell looks under
`/kaggle/input` for `<category>/train/good` before downloading.

Each category has a `train/good` folder (defect-free only) and a `test` folder with good parts and several defect types.
The model **never sees a defective part during training.**

In [ ]:
MVTEC_URLS = {  # official per-category links, https://www.mvtec.com/company/research/datasets/mvtec-ad/downloads
    "bottle":    "https://www.mydrive.ch/shares/163640/9a565b9a02f45cd135e46b1c36e01ec5/download/420937370-1629958698/bottle.tar.xz",
    "cable":     "https://www.mydrive.ch/shares/163641/97fd0295152453e533cfb3db1d253a35/download/420937413-1629958794/cable.tar.xz",
    "capsule":   "https://www.mydrive.ch/shares/163642/236d4304eaf9526b350b18807789ce86/download/420937454-1629958872/capsule.tar.xz",
    "carpet":    "https://www.mydrive.ch/shares/163643/70e8bebbe9f2d2c55043311a4e67eb1d/download/420937484-1629959013/carpet.tar.xz",
    "grid":      "https://www.mydrive.ch/shares/163644/8c7fc7cba43ef5275c87ca750a6d0004/download/420937487-1629959044/grid.tar.xz",
    "hazelnut":  "https://www.mydrive.ch/shares/163645/8346f81c9e093d2ebbb3a73f98372217/download/420937545-1629959162/hazelnut.tar.xz",
    "leather":   "https://www.mydrive.ch/shares/163646/7e0627b5718d2902bb85d24cff7a51ce/download/420937607-1629959262/leather.tar.xz",
    "metal_nut": "https://www.mydrive.ch/shares/163647/6d5a833cdee9035a45d5ce0921ecd24d/download/420937637-1629959294/metal_nut.tar.xz",
    "pill":      "https://www.mydrive.ch/shares/163648/f8c4ca0d325888f2ba128a9ed44708f5/download/420938129-1629960351/pill.tar.xz",
    "screw":     "https://www.mydrive.ch/shares/163649/cfd394b4e3a49013745fab9e2084988a/download/420938130-1629960389/screw.tar.xz",
    "tile":      "https://www.mydrive.ch/shares/163650/55ad2038770deff6fe6771bd784af8cc/download/420938133-1629960456/tile.tar.xz",
    "toothbrush":"https://www.mydrive.ch/shares/163651/f03d8421ec64b3e3bfa4a2ba2fbafb4f/download/420938134-1629960477/toothbrush.tar.xz",
    "transistor":"https://www.mydrive.ch/shares/163652/619a88c3d5f3f57b46b00f44f04ad6e6/download/420938166-1629960554/transistor.tar.xz",
    "wood":      "https://www.mydrive.ch/shares/163653/937e0d65bc9c4497d253cb7313868a60/download/420938383-1629960649/wood.tar.xz",
    "zipper":    "https://www.mydrive.ch/shares/163654/7f6e6aefbf00a9e8a09cc0bbac236eb0/download/420938385-1629960680/zipper.tar.xz",
}

def find_existing(cat):
    """Return a folder containing <cat>/train/good, from DATA_ROOT or a Kaggle input, else None."""
    cands = [os.path.join(DATA_ROOT, cat)]
    for depth in range(1, 5):
        cands += glob.glob(os.path.join("/kaggle/input", *(["*"] * depth), cat))
    for c in cands:
        if os.path.isdir(os.path.join(c, "train", "good")):
            return c
    return None

def download_category(cat, retries=3):
    os.makedirs(DATA_ROOT, exist_ok=True)
    url, tmp = MVTEC_URLS[cat], os.path.join(DATA_ROOT, f"{cat}.tar.xz.part")
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (23N Field Tests)"})
            with urllib.request.urlopen(req, timeout=60) as r, open(tmp, "wb") as f:
                total, done, next_pct = int(r.headers.get("Content-Length", 0)), 0, 25
                while chunk := r.read(1 << 20):
                    f.write(chunk); done += len(chunk)
                    if total and done * 100 // total >= next_pct:
                        print(f"  {cat}: {done/1e6:.0f}/{total/1e6:.0f} MB"); next_pct += 25
            with tarfile.open(tmp, "r:xz") as t:
                try:
                    t.extractall(DATA_ROOT, filter="data")
                except TypeError:          # Python < 3.12
                    t.extractall(DATA_ROOT)
            os.remove(tmp)
            return os.path.join(DATA_ROOT, cat)
        except Exception as e:
            print(f"  {cat}: attempt {attempt} failed: {e}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Could not download {cat}. Download it manually from "
                       "https://www.mvtec.com/company/research/datasets/mvtec-ad/downloads and extract into DATA_ROOT.")

def list_category(cat_dir):
    train = sorted(glob.glob(os.path.join(cat_dir, "train", "good", "*.png")))
    test = []
    for dtype in sorted(os.listdir(os.path.join(cat_dir, "test"))):
        for p in sorted(glob.glob(os.path.join(cat_dir, "test", dtype, "*.png"))):
            stem = os.path.splitext(os.path.basename(p))[0]
            mask = os.path.join(cat_dir, "ground_truth", dtype, f"{stem}_mask.png")
            test.append({"path": p, "label": int(dtype != "good"), "defect": dtype,
                         "mask": mask if os.path.exists(mask) else None})
    return train, test

DATA = {}
t0 = time.time()
for cat in CATEGORIES:
    cat_dir = find_existing(cat)
    if cat_dir is None:
        print(f"Downloading {cat} ...")
        cat_dir = download_category(cat)
    train, test = list_category(cat_dir)
    rng = random.Random(SEED)
    shuffled = train[:]; rng.shuffle(shuffled)
    n_cal = max(10, int(len(shuffled) * CALIB_FRACTION))
    DATA[cat] = {"dir": cat_dir, "train": sorted(shuffled[n_cal:]), "calib": sorted(shuffled[:n_cal]), "test": test}
    n_def = sum(t["label"] for t in test)
    print(f"{cat:10s} train(good)={len(DATA[cat]['train'])}  calib(good)={n_cal}  "
          f"test={len(test)} ({n_def} defective, {len(test)-n_def} good, "
          f"{len(set(t['defect'] for t in test))-1} defect types)")
print(f"Data ready in {time.time()-t0:.0f}s")

In [ ]:
# A look at the data: one good part and two defective parts per category
fig, axes = plt.subplots(len(CATEGORIES), 3, figsize=(9, 3 * len(CATEGORIES)))
for r, cat in enumerate(CATEGORIES):
    good = [t for t in DATA[cat]["test"] if t["label"] == 0][0]
    bad = [t for t in DATA[cat]["test"] if t["label"] == 1]
    bad = random.Random(SEED).sample(bad, 2)
    for c, item in enumerate([good] + bad):
        axes[r, c].imshow(Image.open(item["path"]).convert("RGB"))
        axes[r, c].set_title(f"{CATEGORY_LABELS[cat]} · {item['defect']}", fontsize=9)
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## 3 · Method — PatchCore with a small backbone

[PatchCore](https://arxiv.org/abs/2106.08265) (Roth et al., CVPR 2022), implemented from scratch in ~80 lines:

1. Pass each **good** training image through an ImageNet-pretrained CNN; take mid-level feature maps (layer2 + layer3),
   smooth them with a 3×3 average pool, and upsample layer3 to layer2's 28×28 grid. Every grid cell becomes a
   "patch descriptor".
2. Keep a representative subset of all good-part patches (greedy k-center **coreset**) as a memory bank.
3. At test time, every patch of a new image looks up its nearest neighbour in the bank. Distance = how *unlike* any
   good part that patch is. The image's **anomaly score is the worst patch**; the 28×28 distance grid, upsampled and
   blurred, is the **heatmap**.
4. The alarm **threshold** is set from held-out *good* training images only (99th percentile), so no defect labels are
   used anywhere before evaluation.

No training loop, no labels, no GPU required at inference.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _resize_crop(img, resample):
    img = img.resize((RESIZE, RESIZE), resample)
    o = (RESIZE - IMG_SIZE) // 2
    return img.crop((o, o, o + IMG_SIZE, o + IMG_SIZE))

def prep(src):
    """Path or PIL image -> (normalised tensor [3,H,W], display uint8 array [H,W,3])."""
    img = src if isinstance(src, Image.Image) else Image.open(src)
    arr = np.asarray(_resize_crop(img.convert("RGB"), Image.BILINEAR))
    t = (torch.from_numpy(arr.copy()).permute(2, 0, 1).float() / 255.0 - MEAN) / STD
    return t, arr

def load_mask(path):
    if path is None:
        return None
    return np.asarray(_resize_crop(Image.open(path).convert("L"), Image.NEAREST)) > 127

class FeatureExtractor(torch.nn.Module):
    """ResNet-family trunk returning layer2 and layer3 feature maps."""
    def __init__(self, arch, weights=BACKBONE_WEIGHTS):
        super().__init__()
        m = getattr(torchvision.models, arch)(weights=weights)
        self.stem = torch.nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1)
        self.layer2, self.layer3 = m.layer2, m.layer3
        self.n_params = sum(p.numel() for p in m.parameters())
        self.eval()
    def forward(self, x):
        f2 = self.layer2(self.stem(x))
        return f2, self.layer3(f2)

def embed(fe, x):
    f2, f3 = fe(x)
    f2, f3 = F.avg_pool2d(f2, 3, 1, 1), F.avg_pool2d(f3, 3, 1, 1)
    f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
    return torch.cat([f2, f3], 1)                       # [B, D, 28, 28]

@torch.no_grad()
def greedy_coreset(feats, n_select, proj_dim=128, seed=SEED):
    """Greedy k-center selection on a random projection (as in the PatchCore paper)."""
    N = feats.shape[0]
    if n_select >= N:
        return torch.arange(N)
    g = torch.Generator().manual_seed(seed)
    P = (torch.randn(feats.shape[1], proj_dim, generator=g) / math.sqrt(proj_dim)).to(feats.device)
    z = feats @ P
    first = int(torch.randint(N, (1,), generator=g))
    idx = [first]
    min_d = ((z - z[first]) ** 2).sum(1)
    for _ in range(n_select - 1):
        i = int(torch.argmax(min_d))
        idx.append(i)
        min_d = torch.minimum(min_d, ((z - z[i]) ** 2).sum(1))
    return torch.tensor(idx)

class PatchCore:
    def __init__(self, arch, device=DEVICE, coreset_ratio=CORESET_RATIO, weights=BACKBONE_WEIGHTS):
        self.arch, self.device, self.ratio = arch, device, coreset_ratio
        self.fe = FeatureExtractor(arch, weights).to(device)
        self.bank, self.threshold = None, None

    def to(self, device):
        self.device = device
        self.fe.to(device)
        if self.bank is not None:
            self.bank = self.bank.to(device)
        return self

    @torch.no_grad()
    def fit(self, paths, batch_size=BATCH_SIZE):
        feats = []
        for i in range(0, len(paths), batch_size):
            x = torch.stack([prep(p)[0] for p in paths[i:i + batch_size]]).to(self.device)
            e = embed(self.fe, x)
            self.grid = e.shape[-2:]
            feats.append(e.permute(0, 2, 3, 1).reshape(-1, e.shape[1]))
        feats = torch.cat(feats)
        n = max(1, int(len(feats) * self.ratio))
        self.bank = feats[greedy_coreset(feats, n).to(self.device)].contiguous()
        self.n_patches_seen = len(feats)
        del feats
        return self

    @torch.no_grad()
    def score_tensor(self, x, bank_chunk=8192):
        """x: [B,3,H,W] normalised. Returns (image scores np[B], patch maps torch[B,h,w] on CPU)."""
        e = embed(self.fe, x.to(self.device))
        B, D, h, w = e.shape
        q = e.permute(0, 2, 3, 1).reshape(B, h * w, D)
        dmin = torch.full((B, h * w), float("inf"), device=self.device)
        for j in range(0, len(self.bank), bank_chunk):
            d = torch.cdist(q, self.bank[j:j + bank_chunk].unsqueeze(0).expand(B, -1, -1))
            dmin = torch.minimum(dmin, d.min(-1).values)
        maps = dmin.view(B, h, w)
        return maps.amax((1, 2)).cpu().numpy(), maps.cpu()

    def score_paths(self, paths, batch_size=BATCH_SIZE):
        scores, maps = [], []
        for i in range(0, len(paths), batch_size):
            s, m = self.score_tensor(torch.stack([prep(p)[0] for p in paths[i:i + batch_size]]))
            scores.append(s); maps.append(m)
        return np.concatenate(scores), torch.cat(maps)

def upsample_map(m, size=IMG_SIZE, sigma=4):
    m = F.interpolate(m[None, None].float(), size=(size, size), mode="bilinear", align_corners=False)
    return TF.gaussian_blur(m, kernel_size=8 * sigma + 1, sigma=sigma)[0, 0].numpy()

def measure_latency(pc, paths, device, threads=CPU_THREADS, warmup=3):
    """Median seconds per image, batch size 1, from PNG on disk to anomaly score (decode + resize + model + k-NN)."""
    if device == "cpu":
        torch.set_num_threads(threads)
    pc.to(device)
    times = []
    for i, p in enumerate(paths[:warmup] + paths):
        t0 = time.perf_counter()
        pc.score_tensor(prep(p)[0][None])
        if device == "cuda":
            torch.cuda.synchronize()
        if i >= warmup:
            times.append(time.perf_counter() - t0)
    return float(np.median(times))

In [ ]:
# Fit both backbones on every category, score the test set, time CPU (+GPU) latency.
RESULTS, SMALL_MODELS, SMALL_TEST = [], {}, {}
t_all = time.time()
for size, arch in BACKBONES.items():
    for cat in CATEGORIES:
        d, t0 = DATA[cat], time.time()
        pc = PatchCore(arch).fit(d["train"])
        calib_scores, _ = pc.score_paths(d["calib"])
        pc.threshold = float(np.quantile(calib_scores, THRESH_QUANTILE))
        test_scores, test_maps = pc.score_paths([t["path"] for t in d["test"]])
        labels = np.array([t["label"] for t in d["test"]])
        flagged = test_scores > pc.threshold
        lat_paths = [t["path"] for t in d["test"]][:: max(1, len(d["test"]) // LATENCY_N)][:LATENCY_N]
        lat_gpu = measure_latency(pc, lat_paths, "cuda") if DEVICE == "cuda" else float("nan")
        lat_cpu = measure_latency(pc, lat_paths, "cpu")
        pc.to(DEVICE)
        row = dict(category=cat, backbone=arch, size=size, params_m=pc.fe.n_params / 1e6,
                   bank=len(pc.bank), auroc=roc_auc_score(labels, test_scores),
                   n_defect=int(labels.sum()), n_good=int((1 - labels).sum()),
                   caught=int((flagged & (labels == 1)).sum()), false_alarms=int((flagged & (labels == 0)).sum()),
                   lat_cpu_s=lat_cpu, lat_gpu_s=lat_gpu, fit_s=time.time() - t0)
        RESULTS.append(row)
        print(f"{BACKBONE_NAMES[arch]:14s} {cat:10s} AUROC={row['auroc']:.3f}  caught {row['caught']}/{row['n_defect']}  "
              f"false alarms {row['false_alarms']}/{row['n_good']}  CPU {lat_cpu*1000:.0f} ms/img  "
              f"bank={row['bank']:,}  ({row['fit_s']:.0f}s)")
        if size == "small":
            SMALL_MODELS[cat] = pc
            SMALL_TEST[cat] = {"scores": test_scores, "maps": test_maps.half(), "labels": labels}
        else:
            del pc
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
print(f"\nAll models done in {(time.time()-t_all)/60:.1f} min")
df = pd.DataFrame(RESULTS)

## 4 · Results — the headline number

* **AUROC** (area under the ROC curve): 1.0 means every defective part scores higher than every good part; 0.5 is a coin
  toss. It is threshold-free.
* **Caught** / **false alarms**: what happens on the test set at the threshold set from held-out *good* parts only.
* **Cost per 1,000 inspections** on the CPU box = (amortised box cost + electricity) per hour × hours of compute needed,
  using the *measured* CPU latency. These are **estimates**; every assumption is a named variable in the Setup cell.

In [ ]:
CPU_BOX_INR_PER_HOUR = (CPU_BOX_COST_INR / (CPU_BOX_LIFE_YEARS * 365 * CPU_BOX_HOURS_PER_DAY)
                        + CPU_BOX_POWER_W / 1000 * TARIFF_INR_PER_KWH)

def inr_per_1000(seconds_per_image, inr_per_hour):
    return inr_per_hour * 1000 * seconds_per_image / 3600

def fmt_ratio(r):
    """Round a large ratio to 2 significant figures (e.g. 20,068 -> '20,000')."""
    return f"{round(r, -int(math.floor(math.log10(r))) + 1):,.0f}" if r >= 100 else f"{r:,.0f}"

def fmt_inr(x):
    return f"₹{x:.3f}" if x < 0.1 else f"₹{x:,.2f}" if x < 10 else f"₹{x:,.0f}"

summary = (df.groupby(["size", "backbone"])
             .agg(params_m=("params_m", "first"), auroc=("auroc", "mean"), caught=("caught", "sum"),
                  n_defect=("n_defect", "sum"), false_alarms=("false_alarms", "sum"), n_good=("n_good", "sum"),
                  lat_cpu_s=("lat_cpu_s", "median"), lat_gpu_s=("lat_gpu_s", "median"))
             .reset_index().set_index("size"))
summary["recall"] = summary["caught"] / summary["n_defect"]
summary["false_alarm_rate"] = summary["false_alarms"] / summary["n_good"]
summary["parts_per_min_cpu"] = 60 / summary["lat_cpu_s"]
summary["inr_per_1000_cpu"] = [inr_per_1000(s, CPU_BOX_INR_PER_HOUR) for s in summary["lat_cpu_s"]]
summary["inr_per_1000_cloud_gpu"] = [inr_per_1000(s, CLOUD_GPU_INR_PER_HOUR) for s in summary["lat_gpu_s"]]

per_cat = df.pivot(index="category", columns="backbone", values="auroc")[list(BACKBONES.values())]
per_cat.columns = [BACKBONE_NAMES[c] + " AUROC" for c in per_cat.columns]
display(per_cat.round(3))
display(summary.round(4))

S, L = summary.loc["small"], summary.loc["large"]
H = {  # every number in the post / video comes from here
    "small_name": BACKBONE_NAMES[BACKBONES["small"]], "large_name": BACKBONE_NAMES[BACKBONES["large"]],
    "small_params_m": S.params_m, "large_params_m": L.params_m,
    "size_ratio": L.params_m / S.params_m,
    "small_auroc": S.auroc, "large_auroc": L.auroc, "auroc_gap": L.auroc - S.auroc,
    "recall_pct": 100 * S.recall, "far_pct": 100 * S.false_alarm_rate,
    "large_recall_pct": 100 * L.recall, "large_far_pct": 100 * L.false_alarm_rate,
    "small_ms_cpu": 1000 * S.lat_cpu_s, "large_ms_cpu": 1000 * L.lat_cpu_s,
    "small_ms_gpu": 1000 * S.lat_gpu_s, "large_ms_gpu": 1000 * L.lat_gpu_s,
    "small_inr_1000": S.inr_per_1000_cpu, "large_inr_1000": L.inr_per_1000_cpu,
    "parts_per_min": S.parts_per_min_cpu,
    "box_inr": CPU_BOX_COST_INR, "n_categories": len(CATEGORIES),
    "n_test": int(S.n_defect + S.n_good), "n_defect": int(S.n_defect),
}
HEADLINE = (f"A {H['small_name']} on a ₹{H['box_inr']:,} CPU box flags {H['recall_pct']:.0f}% of defects "
            f"at {fmt_inr(H['small_inr_1000'])} per 1,000 parts.")
print("\nHEADLINE:", HEADLINE)
print(f"  how: trained on good parts only; mean AUROC {H['small_auroc']:.3f} vs {H['large_auroc']:.3f} for "
      f"{H['large_name']} ({H['large_params_m']:.0f}M params); false alarms {H['far_pct']:.0f}% of good parts; "
      f"{H['small_ms_cpu']:.0f} ms/part on {CPU_THREADS} CPU threads (estimate: box ₹{CPU_BOX_INR_PER_HOUR:.2f}/h).")

In [ ]:
# AUROC chart (also used in the video)
def auroc_chart(per_cat, path):
    fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=BG)
    ax.set_facecolor(BG)
    y = np.arange(len(per_cat))
    for k, (col, colr) in enumerate(zip(per_cat.columns, [ACCENT, TEAL])):
        ax.barh(y + (k - 0.5) * 0.38, per_cat[col], height=0.36, color=colr, label=col.replace(" AUROC", ""))
        for yi, v in zip(y, per_cat[col]):
            ax.text(v + 0.005, yi + (k - 0.5) * 0.38, f"{v:.3f}", va="center", color=FG, fontsize=9)
    ax.set_yticks(y, [CATEGORY_LABELS[c] for c in per_cat.index], color=FG)
    ax.set_xlim(0.5, 1.06); ax.tick_params(colors=MUTED)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xlabel("Image-level AUROC (1.0 = perfect)", color=MUTED)
    ax.legend(frameon=False, labelcolor=FG, loc="lower right")
    ax.set_title("Defect detection quality, small vs large backbone", color=FG, loc="left")
    fig.tight_layout(); fig.savefig(path, dpi=150, facecolor=BG); plt.show()
auroc_chart(per_cat, os.path.join(OUT_DIR, "ft02_auroc.png"))

### Heatmaps — where the small model thinks something is wrong

In [ ]:
def overlay(rgb, amap, vmin, vmax, alpha=0.55):
    """Blend an anomaly map (jet-like 'inferno') over an RGB uint8 image."""
    a = np.clip((amap - vmin) / max(vmax - vmin, 1e-6), 0, 1)
    heat = (plt.get_cmap("inferno")(a)[..., :3] * 255)
    w = alpha * a[..., None]                       # only hot regions are tinted
    return (rgb * (1 - w) + heat * w).astype(np.uint8)

def pick_examples(cat, n_bad=3, n_good=1, seed=SEED):
    test = DATA[cat]["test"]
    rng = random.Random(seed)
    bad = rng.sample([i for i, t in enumerate(test) if t["label"] == 1], n_bad)
    good = rng.sample([i for i, t in enumerate(test) if t["label"] == 0], n_good)
    return bad + good

TILES = []   # (raw uint8, overlay uint8, mask, caption, score, threshold, label) — reused by video & cover
for cat in CATEGORIES:
    pc, st = SMALL_MODELS[cat], SMALL_TEST[cat]
    all_maps = st["maps"].float()
    vmin, vmax = float(np.quantile(all_maps.numpy(), 0.50)), float(all_maps.max()) * 0.9
    for i in pick_examples(cat):
        item = DATA[cat]["test"][i]
        _, rgb = prep(item["path"])
        amap = upsample_map(all_maps[i])
        TILES.append(dict(cat=cat, raw=rgb, over=overlay(rgb, amap, vmin, vmax), mask=load_mask(item["mask"]),
                          caption=f"{CATEGORY_LABELS[cat]} · {item['defect'].replace('_', ' ')}",
                          score=float(st["scores"][i]), thr=pc.threshold, label=item["label"]))

fig, axes = plt.subplots(len(CATEGORIES), 4, figsize=(12, 3.1 * len(CATEGORIES)), facecolor=BG)
for ax, t in zip(axes.flat, TILES):
    ax.imshow(t["over"])
    if t["mask"] is not None and t["mask"].any():
        ax.contour(t["mask"], levels=[0.5], colors=[TEAL], linewidths=1)
    verdict = "FLAGGED" if t["score"] > t["thr"] else "passed"
    ax.set_title(f"{t['caption']}\nscore {t['score']:.2f} (thr {t['thr']:.2f}) → {verdict}", color=FG, fontsize=8)
    ax.axis("off")
fig.suptitle(f"{H['small_name']} anomaly maps (teal outline = ground-truth defect)", color=FG)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "ft02_heatmaps.png"), dpi=120, facecolor=BG); plt.show()

## 5 · OPTIONAL — frontier vision model, zero-shot (needs your own API key)

Skipped automatically unless `ANTHROPIC_API_KEY` is set (Colab: 🔑 *Secrets* panel → add `ANTHROPIC_API_KEY` and enable
notebook access; Kaggle: *Add-ons → Secrets*). It sends **~40 images** (balanced good/defective across the 4 categories)
to Claude with a one-word "DEFECTIVE or GOOD?" prompt, and reports accuracy and **cost per 1,000 from the actual token
usage**. Expected spend at list prices is well under US$0.50. The small model is scored on the *same* 40 images at its
own threshold for an apples-to-apples comparison.

This is a deliberately simple zero-shot baseline: the API model sees no reference images and no training data,
whereas PatchCore saw ~200–300 good parts per product line.

In [ ]:
def get_api_key():
    k = os.environ.get("ANTHROPIC_API_KEY")
    if k:
        return k
    try:
        from google.colab import userdata
        return userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    except Exception:
        return None

CLAUDE = None
API_KEY = get_api_key()
if not API_KEY:
    print("ANTHROPIC_API_KEY not set → skipping the optional frontier-model comparison. "
          "Everything else (results, video, post) still runs.")
else:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    rng = random.Random(SEED)
    per = max(1, CLAUDE_SAMPLE_N // (2 * len(CATEGORIES)))
    sample = []
    for cat in CATEGORIES:
        test = DATA[cat]["test"]
        for lab in (1, 0):
            idx = [i for i, t in enumerate(test) if t["label"] == lab]
            sample += [(cat, i) for i in rng.sample(idx, min(per, len(idx)))]

    def to_b64_jpeg(path):
        img = Image.open(path).convert("RGB")
        img.thumbnail((CLAUDE_MAX_SIDE, CLAUDE_MAX_SIDE))
        buf = io.BytesIO(); img.save(buf, format="JPEG", quality=90)
        return base64.standard_b64encode(buf.getvalue()).decode()

    rows = []
    for cat, i in sample:
        item = DATA[cat]["test"][i]
        prompt = (f"You are a visual quality inspector on a factory line. The photo shows one "
                  f"{CATEGORY_LABELS[cat].lower().rstrip('s')} from a production batch. Is this part defective "
                  "(scratch, crack, dent, contamination, missing or bent piece, wrong colour, or any other "
                  "manufacturing flaw)? Answer with exactly one word: DEFECTIVE or GOOD.")
        t0 = time.perf_counter()
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=16,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                                 "data": to_b64_jpeg(item["path"])}},
                    {"type": "text", "text": prompt}]}])
            text = "".join(b.text for b in resp.content if b.type == "text").upper()
            pred = 1 if "DEFECT" in text else 0 if "GOOD" in text else None
            rows.append(dict(cat=cat, idx=i, label=item["label"], pred=pred, raw=text.strip(),
                             tok_in=resp.usage.input_tokens, tok_out=resp.usage.output_tokens,
                             latency_s=time.perf_counter() - t0))
        except Exception as e:
            print("API error:", e)
            rows.append(dict(cat=cat, idx=i, label=item["label"], pred=None, raw=str(e)[:80],
                             tok_in=0, tok_out=0, latency_s=np.nan))
    cdf = pd.DataFrame(rows)
    ok = cdf[cdf.pred.notna()]
    cdf["small_pred"] = [int(SMALL_TEST[c]["scores"][i] > SMALL_MODELS[c].threshold) for c, i in zip(cdf.cat, cdf.idx)]
    usd_per_img = (ok.tok_in.mean() * CLAUDE_PRICE_IN_USD_PER_MTOK + ok.tok_out.mean() * CLAUDE_PRICE_OUT_USD_PER_MTOK) / 1e6
    CLAUDE = dict(model=CLAUDE_MODEL, n=len(ok), n_err=int(cdf.pred.isna().sum()),
                  acc=float((ok.pred == ok.label).mean()),
                  recall=float(((ok.pred == 1) & (ok.label == 1)).sum() / max(1, (ok.label == 1).sum())),
                  far=float(((ok.pred == 1) & (ok.label == 0)).sum() / max(1, (ok.label == 0).sum())),
                  small_acc_same=float((cdf.loc[ok.index, "small_pred"] == ok.label).mean()),
                  inr_1000=usd_per_img * 1000 * USD_TO_INR, s_per_img=float(ok.latency_s.median()),
                  tok_in=float(ok.tok_in.mean()))
    H["claude_ratio"] = CLAUDE["inr_1000"] / H["small_inr_1000"]
    display(cdf.head(10))
    print(f"\n{CLAUDE_MODEL}, zero-shot on {CLAUDE['n']} images: accuracy {CLAUDE['acc']:.0%} "
          f"(catches {CLAUDE['recall']:.0%} of defects, false alarms {CLAUDE['far']:.0%})")
    print(f"{H['small_name']} on the same images: accuracy {CLAUDE['small_acc_same']:.0%}")
    print(f"Cost per 1,000: {fmt_inr(CLAUDE['inr_1000'])} (API, ~{CLAUDE['tok_in']:.0f} input tokens/img) vs "
          f"{fmt_inr(H['small_inr_1000'])} (CPU box) → ~{fmt_ratio(H['claude_ratio'])}×")

## 6 · Interactive demo — upload a part, get a score and a heatmap

Uses the small (ResNet-18) models fitted above. Pick the product line, upload a photo (or click an example).
Note: the model only knows what *good* parts of that product line look like under MVTec's lighting and framing;
photos from a phone camera will look "anomalous" for that reason alone.

In [ ]:
try:
    import gradio as gr

    def inspect(image, cat):
        if image is None:
            return None, "Upload an image first."
        pc, st = SMALL_MODELS[cat], SMALL_TEST[cat]
        t, rgb = prep(Image.fromarray(image))
        t0 = time.perf_counter()
        s, m = pc.score_tensor(t[None])
        ms = (time.perf_counter() - t0) * 1000
        all_maps = st["maps"].float()
        over = overlay(rgb, upsample_map(m[0]), float(np.quantile(all_maps.numpy(), 0.5)), float(all_maps.max()) * 0.9)
        verdict = "⚠️ LIKELY DEFECTIVE" if s[0] > pc.threshold else "✅ looks like a good part"
        return over, (f"{verdict}\nanomaly score {s[0]:.2f} (alarm threshold {pc.threshold:.2f}) · "
                      f"{ms:.0f} ms on {pc.device.upper()}")

    examples = []
    for cat in CATEGORIES:
        test = DATA[cat]["test"]
        examples += [[next(t["path"] for t in test if t["label"] == 1), cat],
                     [next(t["path"] for t in test if t["label"] == 0), cat]]

    demo = gr.Interface(
        fn=inspect,
        inputs=[gr.Image(type="numpy", label="Part photo"),
                gr.Dropdown(CATEGORIES, value=CATEGORIES[0], label="Product line")],
        outputs=[gr.Image(label="Anomaly heatmap"), gr.Textbox(label="Verdict")],
        examples=examples,
        title="23N Field Test #02 · Small Model, Big Factory",
        description="PatchCore + ResNet-18 trained on good parts only (MVTec AD, CC BY-NC-SA 4.0, research use).")
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo skipped:", e)

## 7 · Video render — 1080×1080, 30 fps, ~58 s, captions burned in

Scenes: hook → the good parts it learned from → heatmaps lighting up on defects → AUROC bars → cost per 1,000 →
headline → end card. Each scene is encoded to its own file and joined with ffmpeg's concat demuxer.
Also writes a 1080×1350 cover image.

In [ ]:
VW = VH = 1080
FPS = 30

def _canvas(w=VW, h=VH):
    fig = Figure(figsize=(w / 100, h / 100), dpi=100, facecolor=BG)
    FigureCanvasAgg(fig)
    return fig

def _rgb(fig):
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

def _bg(w=VW, h=VH):
    return np.broadcast_to(np.array(matplotlib.colors.to_rgb(BG)) * 255, (h, w, 3)).astype(np.float32)

def _ease(x):
    x = min(max(x, 0.0), 1.0)
    return 1 - (1 - x) ** 3

def _brand(fig, caption=None):
    fig.text(0.05, 0.955, "23N FIELD TESTS · #02", color=ACCENT, fontsize=17, weight="bold", va="center")
    fig.text(0.95, 0.955, "Small Model, Big Factory", color=MUTED, fontsize=15, ha="right", va="center")
    if caption:
        fig.patches.append(matplotlib.patches.Rectangle((0, 0), 1, 0.105, transform=fig.transFigure,
                                                        color="#151B23", zorder=-1))
        fig.text(0.5, 0.0525, caption, color=FG, fontsize=23, ha="center", va="center", wrap=True)

class _Scene:
    """Pipe raw RGB frames into one ffmpeg process; rename the file only if ffmpeg exits cleanly."""
    def __init__(self, path):
        self.path, self.tmp = path, path[:-4] + ".tmp.mp4"
        self.p = subprocess.Popen(
            ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{VW}x{VH}",
             "-r", str(FPS), "-i", "-", "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
             "-pix_fmt", "yuv420p", "-r", str(FPS), self.tmp], stdin=subprocess.PIPE)
        self.n = 0
    def add(self, frame, repeat=1):
        b = np.ascontiguousarray(frame, dtype=np.uint8).tobytes()
        for _ in range(repeat):
            self.p.stdin.write(b); self.n += 1
    def close(self):
        self.p.stdin.close()
        if self.p.wait() != 0:
            raise RuntimeError(f"ffmpeg failed on {self.path}")
        os.replace(self.tmp, self.path)
        return self.n

def _fade_scene(path, full, seconds, fade_in=0.6, fade_out=0.0):
    """Static frame with fade from / to background."""
    bg, full = _bg(), full.astype(np.float32)
    sc, n = _Scene(path), int(round(seconds * FPS))
    for k in range(n):
        t = k / FPS
        a = min(1.0, t / fade_in) if fade_in else 1.0
        if fade_out:
            a = min(a, (seconds - t) / fade_out)
        sc.add(full if a >= 1 else bg + (full - bg) * max(a, 0))
    return sc.close()

def _reveal_scene(path, base, full, boxes, seconds, start=0.3, stagger=0.35, ramp=0.8, base_fade=0.0):
    """Blend `full` over `base` region by region (boxes in pixel coords), staggered in time."""
    base, full, bg = base.astype(np.float32), full.astype(np.float32), _bg()
    sc, n = _Scene(path), int(round(seconds * FPS))
    for k in range(n):
        t = k / FPS
        frame = base.copy() if not base_fade else bg + (base - bg) * min(1.0, t / base_fade)
        for j, (x0, y0, x1, y1) in enumerate(boxes):
            a = _ease((t - start - j * stagger) / ramp)
            if a > 0:
                frame[y0:y1, x0:x1] = frame[y0:y1, x0:x1] * (1 - a) + full[y0:y1, x0:x1] * a
        sc.add(frame)
    return sc.close()

def _ax_box(fig, ax):
    bb = ax.get_window_extent()
    return int(bb.x0), int(VH - bb.y1), int(math.ceil(bb.x1)), int(math.ceil(VH - bb.y0))

def _grid_axes(fig, rows, cols, top=0.86, bottom=0.14, left=0.05, right=0.95, gap=0.012, label_h=0.03):
    cw = (right - left - gap * (cols - 1)) / cols
    ch = (top - bottom - (gap + label_h) * (rows - 1)) / rows - label_h
    side = min(cw, ch)
    x_off = left + (right - left - (side * cols + gap * (cols - 1))) / 2
    axes = []
    for r in range(rows):
        for c in range(cols):
            x = x_off + c * (side + gap)
            y = top - (r + 1) * side - r * (gap + label_h) - label_h * r
            ax = fig.add_axes([x, y, side, side]); ax.axis("off")
            axes.append(ax)
    return axes

def make_video(V, out_path="ft02_video.mp4", workdir=None):
    """V: dict of computed assets/numbers (see build_video_assets). Returns path of the joined mp4."""
    workdir = workdir or os.path.join(OUT_DIR, "_scenes"); os.makedirs(workdir, exist_ok=True)
    scenes = []

    # S1 hook (0–5 s)
    fig = _canvas(); _brand(fig)
    fig.text(0.5, 0.60, "Do factories need", color=FG, fontsize=64, ha="center", weight="bold")
    fig.text(0.5, 0.50, "giant AI models", color=ACCENT, fontsize=72, ha="center", weight="bold")
    fig.text(0.5, 0.40, "to spot a defect?", color=FG, fontsize=64, ha="center", weight="bold")
    fig.text(0.5, 0.24, V["hook_sub"], color=MUTED, fontsize=26, ha="center")
    scenes.append(os.path.join(workdir, "s1.mp4")); _fade_scene(scenes[-1], _rgb(fig), 5.0)

    # S2 good parts (5–11 s)
    goods = V["good_images"]
    fig = _canvas(); _brand(fig, "Trained ONLY on photos of good parts. No defect labels.")
    fig.text(0.5, 0.885, "What the model learned from", color=FG, fontsize=36, ha="center", weight="bold")
    axes = _grid_axes(fig, 2, 2, top=0.80, bottom=0.13, label_h=0.04)
    base = _rgb(fig)
    for ax, (label, img) in zip(axes, goods):
        ax.imshow(img); ax.set_title(label, color=FG, fontsize=22, pad=6)
    full = _rgb(fig)
    boxes = []
    for ax in axes:
        x0, y0, x1, y1 = _ax_box(fig, ax); boxes.append((x0, max(0, y0 - 45), x1, y1))
    scenes.append(os.path.join(workdir, "s2.mp4"))
    _reveal_scene(scenes[-1], base, full, boxes, 6.0, start=0.2, stagger=0.4, ramp=0.6, base_fade=0.3)

    # S3 heatmaps (11–24 s): raw defective parts appear, then heat lights up tile by tile
    tiles = V["tiles"]
    cols = 4 if len(tiles) > 9 else 3
    rows = int(math.ceil(len(tiles) / cols))
    fig = _canvas(); _brand(fig, "Bright = unlike any good part. It found these on its own.")
    fig.text(0.5, 0.885, f"{V['small_name']} on a CPU, looking for trouble", color=FG, fontsize=32,
             ha="center", weight="bold")
    axes = _grid_axes(fig, rows, cols, top=0.83, bottom=0.115, gap=0.01, label_h=0.042)
    ims = []
    for ax, t in zip(axes, tiles):
        ims.append(ax.imshow(t["raw"])); ax.set_title(t["caption"].replace(" · ", "\n"), color=MUTED, fontsize=14, pad=4, linespacing=1.1)
    raw = _rgb(fig)
    for im, ax, t in zip(ims, axes, tiles):
        im.set_data(t["over"])
        if t.get("mask") is not None and np.any(t["mask"]):
            ax.contour(t["mask"], levels=[0.5], colors=[TEAL], linewidths=1.5)
    heat = _rgb(fig)
    boxes = [_ax_box(fig, ax) for ax in axes[:len(tiles)]]
    tmp_raw = os.path.join(workdir, "s3a.mp4")
    _fade_scene(tmp_raw, raw, 2.5, fade_in=0.5)
    tmp_heat = os.path.join(workdir, "s3b.mp4")
    _reveal_scene(tmp_heat, raw, heat, boxes, 10.5, start=0.2, stagger=min(0.6, 7.0 / max(1, len(tiles))), ramp=0.7)
    scenes += [tmp_raw, tmp_heat]

    # S4 AUROC bars (24–32 s)
    fig = _canvas(); _brand(fig, "AUROC: 1.0 = every defective part ranked above every good one.")
    fig.text(0.5, 0.885, "Small vs large model: detection quality", color=FG, fontsize=34, ha="center", weight="bold")
    ax = fig.add_axes([0.27, 0.27, 0.62, 0.55]); ax.set_facecolor(BG)
    cats = V["auroc_labels"]; y = np.arange(len(cats))[::-1]
    series = [(V["small_label"], V["auroc_small"], ACCENT), (V["large_label"], V["auroc_large"], TEAL)]
    bars, texts = [], []
    for k, (lab, vals, colr) in enumerate(series):
        off = (0.5 - k) * 0.4
        b = ax.barh(y + off, [0.5] * len(vals), left=0, height=0.38, color=colr, label=lab)
        bars.append((b, vals))
        texts.append([ax.text(0.5, yi + off, "", va="center", color=FG, fontsize=20) for yi in y])
    ax.set_yticks(y, cats, color=FG, fontsize=24); ax.set_xlim(0.5, 1.08)
    ax.set_xticks([0.5, 0.75, 1.0], ["0.5\ncoin toss", "0.75", "1.0\nperfect"], color=MUTED, fontsize=16)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
    ax.legend(loc="upper center", bbox_to_anchor=(0.35, -0.16), ncol=2, frameon=False, labelcolor=FG, fontsize=20)
    sc = _Scene(os.path.join(workdir, "s4.mp4")); scenes.append(sc.path)
    n = 8 * FPS
    for k in range(n):
        a = _ease((k / FPS - 0.3) / 2.5)
        for (b, vals), tx in zip(bars, texts):
            for rect, v, t in zip(b, vals, tx):
                w = 0.5 + (v - 0.5) * a
                rect.set_width(w)
                t.set_x(w + 0.008); t.set_text(f"{v:.3f}" if a > 0.98 else "")
        sc.add(_rgb(fig))
    sc.close()

    # S5 cost per 1,000 (32–40 s), log scale
    fig = _canvas(); _brand(fig, V["cost_caption"])
    fig.text(0.5, 0.885, "Cost per 1,000 inspections (estimate)", color=FG, fontsize=34, ha="center", weight="bold")
    ax = fig.add_axes([0.08, 0.2, 0.84, 0.58]); ax.set_facecolor(BG); ax.axis("off")
    items = V["cost_items"]            # list of (label, inr, sublabel, colour)
    lo = math.log10(min(v for _, v, _, _ in items)) - 0.6
    hi = math.log10(max(v for _, v, _, _ in items)) + 0.15
    ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(items) - 0.5)
    rects, vals_txt = [], []
    for j, (lab, v, sub, colr) in enumerate(items):
        yj = len(items) - 1 - j
        ax.text(0, yj + 0.3, lab, color=FG, fontsize=26, va="center", weight="bold")
        ax.text(0, yj - 0.33, sub, color=MUTED, fontsize=18, va="center")
        r = matplotlib.patches.Rectangle((0, yj - 0.12), 0.0, 0.26, color=colr); ax.add_patch(r)
        rects.append((r, (math.log10(v) - lo) / (hi - lo)))
        vals_txt.append(ax.text(0, yj, "", color=FG, fontsize=26, va="center", weight="bold"))
    ax.text(1.0, -0.62, "bar length on log scale", color=MUTED, fontsize=14, ha="right")
    sc = _Scene(os.path.join(workdir, "s5.mp4")); scenes.append(sc.path)
    for k in range(8 * FPS):
        for j, ((r, target), tx) in enumerate(zip(rects, vals_txt)):
            a = _ease((k / FPS - 0.3 - 0.6 * j) / 1.6)
            r.set_width(max(0.004, 0.78 * target * a))
            tx.set_x(0.78 * target * a + 0.015)
            tx.set_text(fmt_inr(items[j][1]) if a > 0.97 else "")
        sc.add(_rgb(fig))
    sc.close()

    # S6 headline (40–52 s)
    fig = _canvas(); _brand(fig, V["how"])
    fig.text(0.5, 0.80, V["headline_top"], color=FG, fontsize=36, ha="center", weight="bold")
    fig.text(0.5, 0.56, V["headline_big"], color=ACCENT, fontsize=190, ha="center", va="center", weight="bold")
    fig.text(0.5, 0.40, V["headline_mid"], color=FG, fontsize=34, ha="center")
    fig.text(0.5, 0.29, V["headline_low"], color=TEAL, fontsize=44, ha="center", weight="bold")
    fig.text(0.5, 0.185, V["headline_note"], color=MUTED, fontsize=22, ha="center", va="center", linespacing=1.4)
    scenes.append(os.path.join(workdir, "s6.mp4")); _fade_scene(scenes[-1], _rgb(fig), 12.0, fade_in=0.5)

    # S7 end card (52–58 s)
    fig = _canvas()
    fig.text(0.5, 0.62, "23N AI Labs", color=FG, fontsize=80, ha="center", weight="bold")
    fig.text(0.5, 0.52, "Field Test #02 · Open notebook in comments", color=ACCENT, fontsize=34, ha="center")
    fig.text(0.5, 0.42, "23nlabs.com", color=TEAL, fontsize=44, ha="center", weight="bold")
    fig.text(0.5, 0.12, "Independent study on public data · MVTec AD (CC BY-NC-SA 4.0, non-commercial)\n"
             "Costs are estimates; assumptions in the notebook.", color=MUTED, fontsize=18, ha="center",
             linespacing=1.5)
    scenes.append(os.path.join(workdir, "s7.mp4")); _fade_scene(scenes[-1], _rgb(fig), 6.0, fade_in=0.5, fade_out=0.6)

    lst = os.path.join(workdir, "list.txt")
    with open(lst, "w") as f:
        f.writelines(f"file '{os.path.abspath(p)}'\n" for p in scenes)
    tmp = out_path[:-4] + ".tmp.mp4"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", lst,
                    "-c", "copy", "-movflags", "+faststart", tmp], check=True)
    os.replace(tmp, out_path)
    return out_path

def make_cover(V, out_path="ft02_cover.png"):
    W2, H2 = 1080, 1350
    fig = _canvas(W2, H2)
    fig.text(0.06, 0.955, "23N FIELD TESTS · #02", color=ACCENT, fontsize=22, weight="bold", va="center")
    fig.text(0.06, 0.905, "Small Model, Big Factory", color=FG, fontsize=50, weight="bold", va="center")
    fig.text(0.06, 0.862, "Do factories need giant AI models to spot a defect?", color=MUTED, fontsize=24, va="center")
    tiles = [t for t in V["tiles"] if t["label"] == 1][:6]
    side, gap, x0, top = 0.29, 0.015, 0.06, 0.825
    for j, t in enumerate(tiles):
        r, c = divmod(j, 3)
        ax = fig.add_axes([x0 + c * (side + gap), top - (r + 1) * side * W2 / H2 - r * gap, side, side * W2 / H2])
        ax.imshow(t["over"]); ax.axis("off")
    fig.text(0.06, 0.27, V["headline_big"], color=ACCENT, fontsize=105, weight="bold", va="center")
    fig.text(0.50, 0.29, V["headline_mid"], color=FG, fontsize=25, va="center")
    fig.text(0.50, 0.25, V["headline_low"], color=TEAL, fontsize=25, va="center", weight="bold")
    fig.text(0.06, 0.145, V["headline_top"] + ".\n" + V["headline_note"] + "\n" + V["how"],
             color=MUTED, fontsize=21, va="center", linespacing=1.5)
    fig.text(0.06, 0.06, "23nlabs.com · independent study on public data (MVTec AD, CC BY-NC-SA 4.0)",
             color=MUTED, fontsize=18, va="center")
    Image.fromarray(_rgb(fig)).save(out_path)
    return out_path

In [ ]:
def build_video_assets():
    goods = []
    for cat in CATEGORIES:
        p = DATA[cat]["train"][0]
        goods.append((CATEGORY_LABELS[cat], prep(p)[1]))
    bad_tiles = [t for t in TILES if t["label"] == 1]
    cost_items = [(f"{H['small_name']} on a CPU box", H["small_inr_1000"], f"{H['small_ms_cpu']:.0f} ms per part", ACCENT),
                  (f"{H['large_name']} on a CPU box", H["large_inr_1000"], f"{H['large_ms_cpu']:.0f} ms per part", TEAL)]
    if CLAUDE:
        cost_items.append((f"Frontier API ({CLAUDE['model']})", CLAUDE["inr_1000"],
                           f"zero-shot, {CLAUDE['acc']:.0%} accuracy on {CLAUDE['n']} parts "
                           f"(small model: {CLAUDE['small_acc_same']:.0%})", MUTED))
        cost_caption = f"Same task via a frontier API: ~{fmt_ratio(H['claude_ratio'])}× the cost per part."
    else:
        cost_caption = f"CPU box ₹{CPU_BOX_COST_INR:,} over {CPU_BOX_LIFE_YEARS} yrs + power, {CPU_BOX_HOURS_PER_DAY} h/day."
    return dict(
        hook_sub=f"An independent test on {H['n_test']} real factory-part photos",
        good_images=goods, tiles=bad_tiles[:12], small_name=H["small_name"],
        auroc_labels=[CATEGORY_LABELS[c] for c in per_cat.index],
        auroc_small=list(per_cat.iloc[:, 0]), auroc_large=list(per_cat.iloc[:, 1]),
        small_label=f"{H['small_name']} ({H['small_params_m']:.0f}M params)",
        large_label=f"{H['large_name']} ({H['large_params_m']:.0f}M)",
        cost_items=cost_items, cost_caption=cost_caption,
        headline_top=f"A {H['small_name']} on a ₹{H['box_inr']:,} CPU box",
        headline_big=f"{H['recall_pct']:.0f}%",
        headline_mid="of defective parts flagged",
        headline_low=f"at {fmt_inr(H['small_inr_1000'])} per 1,000 parts",
        headline_note=(f"Mean AUROC {H['small_auroc']:.3f} vs {H['large_auroc']:.3f} for a model {H['size_ratio']:.0f}× bigger\n"
                       f"False alarms on {H['far_pct']:.0f}% of good parts"),
        how="How: PatchCore trained on good parts only, CPU inference.",
    )

V = build_video_assets()
t0 = time.time()
VIDEO = make_video(V, "ft02_video.mp4")
COVER = make_cover(V, "ft02_cover.png")
probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                        "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO],
                       capture_output=True, text=True)
print(f"Rendered {VIDEO} in {time.time()-t0:.0f}s:", json.loads(probe.stdout))
try:
    from google.colab import files
    files.download(VIDEO); files.download(COVER)
except Exception:
    print("Not on Colab: find ft02_video.mp4 and ft02_cover.png in the working directory.")

## 8 · Post text — filled with this run's numbers, ready to paste

In [ ]:
POST_LINKEDIN = """Do factories need giant AI models to spot a defect?

23N Field Test #02. We trained a small open model on photos of GOOD parts only: metal nuts, screws, glass bottles and power cables from the public MVTec AD dataset. No defect labels.

Result: {headline}

How it works: a {small_name} ({small_params_m:.0f}M params) turns each photo into patch features; anything unlike every good patch lights up on a heatmap. Mean AUROC {small_auroc:.3f} vs {large_auroc:.3f} for a {large_name} ~{size_ratio:.0f}x its size. {small_ms_cpu:.0f} ms per part on a plain CPU, false alarms on {far_pct:.0f}% of good parts.
{claude_line}
Why it matters: MSME lines in India can run inspection on a ₹{box_inr:,} box on the shop floor, with no cloud bill and no data leaving the plant.

Limits: lab photos, 4 product lines, a fixed camera; costs are estimates (assumptions in the notebook). MVTec AD is CC BY-NC-SA 4.0, research use only. Independent study on public data.

Open notebook + video in comments. Runs free on Colab.

#ManufacturingAI #EdgeAI #MakeInIndia #ComputerVision #23NFieldTests"""

POST_X = """Do factories need giant AI models to spot a defect?

{headline}

Trained on good parts only (MVTec AD). AUROC {small_auroc:.3f} vs {large_auroc:.3f} for a {size_ratio:.0f}x bigger model.{claude_short}

Open notebook ↓ #EdgeAI"""

POST_X_THREAD = [
    "1/ Method: PatchCore. ImageNet {small_name} features per image patch, a memory bank of good-part patches, nearest-neighbour distance = anomaly score. No training loop, no defect labels.",
    "2/ Alarm threshold set from held-out GOOD parts only. On {n_test} test photos it caught {recall_pct:.0f}% of defects with {far_pct:.0f}% false alarms. {small_ms_cpu:.0f} ms/part on CPU.",
    "3/ Caveats: lab photos, fixed camera, 4 product lines, cost = estimate. Data is CC BY-NC-SA (research only). Notebook runs free on Colab: {colab}",
]

COLAB_URL = "https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft02-small-model-defects/ft02_small_model_defects.ipynb"
vals = dict(H, headline=HEADLINE, colab=COLAB_URL)
if CLAUDE:
    vals["claude_line"] = (f"\nFor contrast, a frontier vision model ({CLAUDE['model']}) asked zero-shot on {CLAUDE['n']} of the same "
                           f"parts: {CLAUDE['acc']:.0%} accuracy (small model: {CLAUDE['small_acc_same']:.0%}) at "
                           f"{fmt_inr(CLAUDE['inr_1000'])} per 1,000, ~{fmt_ratio(H['claude_ratio'])}x the cost.\n")
    vals["claude_short"] = f" A frontier API, zero-shot: {CLAUDE['acc']:.0%} accuracy at ~{fmt_ratio(H['claude_ratio'])}x the cost."
else:
    vals["claude_line"] = ""
    vals["claude_short"] = ""

li = POST_LINKEDIN.format(**vals)
x = POST_X.format(**vals)
if len(x) > 280:   # drop the optional clause first, then the hashtag
    x = POST_X.format(**dict(vals, claude_short=""))
thread = [t.format(**vals) for t in POST_X_THREAD]
for name, text, limit in [("LINKEDIN", li, 1300), ("X", x, 280)] + [(f"X THREAD {i+1}", t, 280) for i, t in enumerate(thread)]:
    flag = "OK" if len(text) <= limit else f"OVER LIMIT by {len(text)-limit} — trim before posting"
    print(f"===== {name} ({len(text)}/{limit} chars, {flag}) =====\n{text}\n")
with open(os.path.join(OUT_DIR, "post_linkedin.txt"), "w") as f: f.write(li)
with open(os.path.join(OUT_DIR, "post_x.txt"), "w") as f: f.write(x + "\n\n" + "\n\n".join(thread))

## 9 · Limitations (read before quoting the numbers)

* **Lab conditions.** MVTec AD photos are taken with a fixed industrial camera, controlled lighting, one part per image,
  centred. Real lines add motion blur, dust on the lens, lighting drift and part-position variation; expect lower
  accuracy until you re-collect good-part images on *your* line.
* **Four product lines, ~500 test images.** Per-category test sets have only 40–90 defective parts, so a single image
  moves recall by 1–2 percentage points. Treat differences of ±0.01 AUROC as noise.
* **Threshold choice drives "% of defects flagged".** We set it from held-out good parts (99th percentile), which keeps
  false alarms low but may miss subtle defects. A plant would tune it to its own cost of a miss vs a false reject.
  AUROC is the threshold-free number.
* **Simplified PatchCore.** No feature-dimension reduction, no score re-weighting, a 224-px centre crop, and a smaller
  coreset on CPU runtimes (`CORESET_RATIO`). Published PatchCore numbers are higher; tiny defects on screws suffer most
  from the 224-px resolution.
* **Latency is measured on the notebook's CPU** (2 vCPUs on free Colab), batch size 1, including PNG decode. A modern
  4-core mini-PC is usually faster; ONNX/OpenVINO export and INT8 would cut it further. We did not test ARM boards.
* **Cost per 1,000 is an estimate**: box price, lifetime, hours of use, power draw, tariff and exchange rate are
  assumptions (Setup cell). It covers compute only, not the camera, lighting, integration or labour.
* **Frontier-model comparison (if run) is a zero-shot baseline on ~40 images**: no reference images, no prompt tuning,
  list prices. A frontier model with good reference images or fine-tuning would do better; the point is the cost and
  setup gap, not a verdict on any vendor.
* **Licence.** MVTec AD is CC BY-NC-SA 4.0: research and non-commercial use only. Any production deployment must be
  trained on your own images. Independent study on public data; no affiliation with MVTec or any manufacturer.

*References:* Bergmann et al., "MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection",
CVPR 2019; Bergmann et al., IJCV 2021; Roth et al., "Towards Total Recall in Industrial Anomaly Detection" (PatchCore),
CVPR 2022.